# Kvasir-VQA x1 — Text-only BERT classifier (top-K answers)

Fine-tune a BERT-style encoder on question text for top-K answer classification, using the same splits and label mapping as the TF-IDF baseline. Outputs mirror existing x1 conventions.


In [1]:
from pathlib import Path
import json
import random
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [2]:
# Paths & config

def find_dataset_root() -> Path:
    start = Path.cwd().resolve()
    for p in [start] + list(start.parents):
        if (p / "0_dataset_prep").exists():
            return p
    raise RuntimeError(f"Could not locate dataset root containing '0_dataset_prep'. cwd={start}")

DATA_ROOT = find_dataset_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "01_text_only" / "out" / "02_bert"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "distilbert-base-uncased"  # lightweight, can switch to bert-base-uncased
TOP_K = 200
MAX_LENGTH = 64
BATCH_SIZE = 16
EPOCHS = 5
LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2
GRAD_CLIP = 1.0

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)


Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/01_text_only/out/02_bert
Device: cuda


In [3]:
# Load metadata and splits
meta = pd.read_csv(META_CSV)

meta["question_norm"] = meta["question"].fillna("").astype(str).str.lower().str.strip()
meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


{'train': 46966, 'val': 5931, 'test': 5952}


In [4]:
# Top-K answers
answer_counts = train_df["answer_norm"].value_counts()
TOP_K_ANSWERS = answer_counts.head(TOP_K).index.tolist()

train_k = train_df[train_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
val_k = val_df[val_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
test_k = test_df[test_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)

print("Top-K answers:", len(TOP_K_ANSWERS))
print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})


Top-K answers: 200
{'train': 46598, 'val': 5884, 'test': 5893}


In [5]:
# Label mapping
classes = sorted(TOP_K_ANSWERS)
class_to_idx = {c: i for i, c in enumerate(classes)}

def encode_labels(df):
    return df["answer_norm"].map(class_to_idx).values

train_labels = encode_labels(train_k)
val_labels = encode_labels(val_k) if len(val_k) else None
test_labels = encode_labels(test_k) if len(test_k) else None

print("#classes:", len(classes))


#classes: 200


In [6]:
# Tokenizer and datasets
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class QAQuizDataset(Dataset):
    def __init__(self, df, labels=None):
        self.texts = df["question_norm"].tolist()
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = QAQuizDataset(train_k, train_labels)
val_ds = QAQuizDataset(val_k, val_labels) if val_labels is not None else None
test_ds = QAQuizDataset(test_k, test_labels) if test_labels is not None else None


In [7]:
# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE) if val_ds else None
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE) if test_ds else None


In [8]:
# Class weights (optional)
class_weights = None
counts = Counter(train_labels)
if len(counts):
    weights = np.zeros(len(classes), dtype=np.float32)
    for c, idx in class_to_idx.items():
        weights[idx] = len(train_labels) / (len(classes) * counts[idx])
    class_weights = torch.tensor(weights, dtype=torch.float32)  # keep on CPU first
    try:
        class_weights = class_weights.to(DEVICE)
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            print("Class weights remain on CPU because GPU is full; will drop weighting.")
            class_weights = None
        else:
            raise
print("Using class weights:", class_weights is not None)


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(classes),
)
model.to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps
)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


In [ ]:
# Training & evaluation helpers

def run_eval(loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**{k: v for k, v in batch.items() if k != "labels"})
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            all_probs.append(probs.cpu())
            all_preds.append(preds.cpu())
            all_labels.append(batch["labels"].cpu())
    y_true = torch.cat(all_labels).numpy()
    y_pred = torch.cat(all_preds).numpy()
    probs = torch.cat(all_probs).numpy()
    # map back to labels
    y_true_lbl = [classes[i] for i in y_true]
    y_pred_lbl = [classes[i] for i in y_pred]
    metrics = {
        "accuracy": float(accuracy_score(y_true_lbl, y_pred_lbl)),
        "macro_f1": float(f1_score(y_true_lbl, y_pred_lbl, average="macro")),
    }
    report = classification_report(y_true_lbl, y_pred_lbl, output_dict=True, zero_division=0)
    return metrics, report, y_pred_lbl, probs


def save_metrics(prefix, split_name, metrics, report, extra=None):
    payload = {"metrics": metrics, "report": report}
    if extra is not None:
        payload["extra"] = extra
    with open(OUT_DIR / f"{prefix}_metrics_{split_name}.json", "w") as f:
        json.dump(payload, f, indent=2)


def save_predictions(prefix, split_name, df, preds):
    out_df = df[["question", "answer", "answer_norm"]].copy()
    out_df["pred"] = preds
    out_df.to_csv(OUT_DIR / f"{prefix}_pred_{split_name}.csv", index=False)


def save_probs(prefix, split_name, probs, ids):
    np.savez(OUT_DIR / f"{prefix}_probs_{split_name}.npz", probs=probs, classes=np.array(classes), ids=np.array(ids))


In [ ]:
# Training loop with early stopping on val macro-F1
best_state = None
best_val_f1 = -1.0
patience = PATIENCE
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss if class_weights is None else criterion(outputs.logits, batch["labels"])
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        epoch_loss += loss.item() * batch["labels"].size(0)
    epoch_loss /= len(train_loader.dataset)

    log = {"epoch": epoch, "train_loss": epoch_loss}

    if val_loader is not None:
        val_metrics, _, _, _ = run_eval(val_loader)
        log.update(val_metrics)
        val_f1 = val_metrics["macro_f1"]
        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1 = val_f1
            best_state = model.state_dict()
            patience = PATIENCE
        else:
            patience -= 1
            if patience == 0:
                print("Early stopping")
                history.append(log)
                break
    history.append(log)
    print(log)

if best_state is not None:
    model.load_state_dict(best_state)


In [ ]:
# Evaluate and save
train_metrics, train_report, train_preds, train_probs = run_eval(train_loader)
save_metrics("bert_text", "train", train_metrics, train_report)
save_predictions("bert_text", "train", train_k, train_preds)
save_probs("bert_text", "train", train_probs, train_k["img_id"].values if "img_id" in train_k.columns else np.arange(len(train_k)))

val_metrics = val_report = val_preds = val_probs = None
if val_loader is not None:
    val_metrics, val_report, val_preds, val_probs = run_eval(val_loader)
    save_metrics("bert_text", "val", val_metrics, val_report)
    save_predictions("bert_text", "val", val_k, val_preds)
    save_probs("bert_text", "val", val_probs, val_k["img_id"].values if "img_id" in val_k.columns else np.arange(len(val_k)))

if test_loader is not None:
    test_metrics, test_report, test_preds, test_probs = run_eval(test_loader)
    save_metrics("bert_text", "test", test_metrics, test_report)
    save_predictions("bert_text", "test", test_k, test_preds)
    save_probs("bert_text", "test", test_probs, test_k["img_id"].values if "img_id" in test_k.columns else np.arange(len(test_k)))
else:
    test_metrics = None

pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)

# Save model & tokenizer
(model_path := OUT_DIR / "bert_text_model").mkdir(parents=True, exist_ok=True)
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print("Saved model to", model_path)


In [ ]:
# Summary table
summary_rows = []
for name, mets in [("train", train_metrics), ("val", val_metrics), ("test", test_metrics)]:
    if mets:
        summary_rows.append({"split": name, "accuracy": mets["accuracy"], "macro_f1": mets["macro_f1"]})

if summary_rows:
    display(pd.DataFrame(summary_rows).set_index("split").round(4))
else:
    print("No metrics collected.")
